In [1]:
import os
import cv2
import numpy as np

def yolo_to_xyxy(label, img_w, img_h):
    cx, cy, w, h = label
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return [x1, y1, x2, y2]

def compute_visibility(boxes):
    visibilities = []

    for i, box_i in enumerate(boxes):
        x1_i, y1_i, x2_i, y2_i = box_i
        area_i = max(1.0, (x2_i - x1_i) * (y2_i - y1_i))
        occluded = 0.0

        for j, box_j in enumerate(boxes):
            if i == j:
                continue

            x1_j, y1_j, x2_j, y2_j = box_j

            xA = max(x1_i, x1_j)
            yA = max(y1_i, y1_j)
            xB = min(x2_i, x2_j)
            yB = min(y2_i, y2_j)

            inter_area = max(0, xB - xA) * max(0, yB - yA)
            occluded += inter_area

        visibility = 1.0 - (occluded / area_i)
        visibility = max(0.05, min(1.0, visibility))
        visibilities.append(visibility)

    return visibilities


In [3]:
# /Volumes/storage_dc/ml_learning/divagar_ws/datasets/coco128/images/train2017/000000000049.jpg
# /Volumes/storage_dc/ml_learning/divagar_ws/datasets/coco128/labels/train2017/000000000049.txt

img = cv2.imread("datasets/coco128/images/train2017/000000000049.jpg")
h, w = img.shape[:2]

labels = []
with open("datasets/coco128/labels/train2017/000000000049.txt") as f:
    for line in f:
        parts = list(map(float, line.strip().split()))
        cls = int(parts[0])
        box = parts[1:]
        labels.append((cls, box))

boxes_xyxy = [yolo_to_xyxy(b[1], w, h) for b in labels]
vis = compute_visibility(boxes_xyxy)

for i, (cls, _) in enumerate(labels):
    print(f"Class {cls}, Visibility: {vis[i]:.2f}")


Class 17, Visibility: 0.73
Class 17, Visibility: 0.71
Class 0, Visibility: 0.05
Class 0, Visibility: 0.05
Class 0, Visibility: 0.05
Class 58, Visibility: 1.00
Class 0, Visibility: 0.21
Class 0, Visibility: 0.05
Class 0, Visibility: 1.00


In [4]:
import cv2

def visualize_visibility(img, boxes, visibilities):
    for box, v in zip(boxes, visibilities):
        x1, y1, x2, y2 = map(int, box)
        color = (0, int(255 * v), int(255 * (1 - v)))  # green → red
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{v:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    cv2.imshow("Visibility Debug", img)
    cv2.waitKey(0)


In [ ]:
visualize_visibility(img, boxes_xyxy, vis)

In [1]:
import glob

vis = []
for f in glob.glob("datasets/coco_occlusion/labels/train2017/*.txt"):
    with open(f) as file:
        for line in file:
            vis.append(float(line.strip().split()[-1]))

print("Min:", min(vis))
print("Max:", max(vis))
print("Mean:", sum(vis) / len(vis))


Min: 0.05
Max: 1.0
Mean: 0.5012161463939752


In [ ]:
import torch
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.model.eval()

dummy = torch.randn(1, 3, 640, 640)

with torch.no_grad():
    preds = model.model(dummy)

print(type(preds))


ImportError: cannot import name '__version__' from 'ultralytics' (unknown location)